# Notebook to explore the Gene Expression data extracted from archs4 H5 file

ref [NIH](https://bioinformatics.ccr.cancer.gov/docs/b4b/RNASeq_Overview/01.Overview/)

Contains documentation for remapping and 
_____

Outputs file path: `/data/FinalExpression/{disease}/expressions(cleaned)/{GSEID}.tsv`

In [38]:
import os
import re
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
#import archs4py as a4
#import plotly.graph_objects as go
import colorsys
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from typing import Dict, Optional

os.getcwd()

'/Users/kewalinsamart/Documents/GitHub/integrative-drugrep-tb/scripts'

## Initialize Variables

#### Do all studies have metadata and expression data?

In [39]:
def metadata_expr_match(metadata_dir, expression_dir):
    """
    Returns True if every GSE in metadata_dir (as metadata_GSE########.tsv)
    matches every GSE in expression_dir (as GSE########.tsv),
    and there are no extra GSEs in either directory.
    """
    meta_ids = {
        f[len("metadata_"):-4]
        for f in os.listdir(metadata_dir)
        if f.startswith("metadata_GSE") and f.endswith(".tsv")
    }
    expr_ids = {
        f[:-4]
        for f in os.listdir(expression_dir)
        if f.startswith("GSE") and f.endswith(".tsv")
    }

    # Conditions:
    # 1. All IDs present in both
    # 2. No extras in either dir
    meta_missing = expr_ids - meta_ids
    expr_missing = meta_ids - expr_ids

    match = meta_ids == expr_ids

    if not match:
        print("Mismatch detected:")
        if meta_missing:
            print(f"Missing in metadata: {sorted(meta_missing)}")
        if expr_missing:
            print(f"Missing in expression: {sorted(expr_missing)}")
    return match


In [69]:
tbdrugrep_meta = "/Users/kewalinsamart/Documents/GitHub/integrative-drugrep-tb/data/microarray_data_forDE/metadata"
tbdrugrep_expr = "/Users/kewalinsamart/Documents/GitHub/integrative-drugrep-tb/data/microarray_data_forDE/expression_matrices/original"
metadata_expr_match(tbdrugrep_meta, tbdrugrep_expr)

True

## Expression Data EDA

In [70]:
def get_expression_shape(path : str, shape : str = None):
    """
    returns the shape of the each file in the directory
    """
    last_dir = os.path.basename(os.path.normpath(path))
    print(f"Number of studies in {last_dir}: {len(os.listdir(path))}")
    
    files = os.listdir(path)
    for file in files:
        if file.endswith('.tsv'):
            df = pd.read_csv(os.path.join(path, file), sep='\t')
            if shape is None:
                print(f"{file}: {df.shape}")
            elif shape == 'rows':
                print(f"{file}: {df.shape[0]} rows")
            elif shape == 'columns':
                print(f"{file}: {df.shape[1]} columns")
            else:
                raise ValueError("Invalid shape argument. Use 'both', 'rows', or 'columns'.")

#### Get Expression File shape

In [71]:
get_expression_shape(tbdrugrep_expr)

Number of studies in original: 12
GSE42834.tsv: (23374, 17)
GSE34151.tsv: (23374, 260)
GSE42830.tsv: (23374, 17)
GSE16250.tsv: (20056, 7)
GSE139871.tsv: (23552, 25)
GSE18794.tsv: (21661, 22)
GSE39941.tsv: (23374, 492)
GSE119143.tsv: (35066, 13)
GSE19491.tsv: (21799, 338)
GSE83456.tsv: (23374, 203)
GSE57028.tsv: (31560, 3)
GSE63548.tsv: (23371, 27)


#### Get list of relevant samples

In [72]:
# read
Relevant_TB = pd.read_csv("/Users/kewalinsamart/Documents/GitHub/integrative-drugrep-tb/data/microarray_data_forDE/clean_TB_sample_metadata_classification.tsv", sep = '\t')

Relevant_TB_samples = set(Relevant_TB["geo_accession"].unique())

print("TB:", len(Relevant_TB_samples))

TB: 723


## Gene Mapping Dictionary Set Up
<a id="GeneMapping"></a>

In [73]:
homo_gene = pd.read_csv("/Users/kewalinsamart/Documents/GitHub/integrative-drugrep-tb/data/metadata/Homo_sapiens.gene_info.tsv", sep = '\t')
homo_gene.head()

,Unnamed: 0,#tax_id,GeneID,Symbol,LocusTag,Synonyms,dbXrefs,chromosome,map_location,description,type_of_gene,Symbol_from_nomenclature_authority,Full_name_from_nomenclature_authority,Nomenclature_status,Other_designations,Modification_date,Feature_type,Ensembl_ID,Ensembl
0,0,9606,1,A1BG,-,A1B|ABG|GAB|HYST2477,MIM:138670|HGNC:HGNC:5|Ensembl:ENSG00000121410...,19,19q13.43,alpha-1-B glycoprotein,protein-coding,A1BG,alpha-1-B glycoprotein,O,alpha-1B-glycoprotein|HEL-S-163pA|epididymis s...,20250819,-,ENSG00000121410,ENSG00000121410
1,1,9606,2,A2M,-,A2MD|CPAMD5|FWP007|S863-7,MIM:103950|HGNC:HGNC:7|Ensembl:ENSG00000175899...,12,12p13.31,alpha-2-macroglobulin,protein-coding,A2M,alpha-2-macroglobulin,O,alpha-2-macroglobulin|C3 and PZP-like alpha-2-...,20250909,-,ENSG00000175899,ENSG00000175899
2,2,9606,9,NAT1,-,AAC1|MNAT|NAT-1|NATI,MIM:108345|HGNC:HGNC:7645|Ensembl:ENSG00000171...,8,8p22,N-acetyltransferase 1,protein-coding,NAT1,N-acetyltransferase 1,O,arylamine N-acetyltransferase 1|N-acetyltransf...,20250909,-,ENSG00000171428,ENSG00000171428
3,3,9606,10,NAT2,-,AAC2|NAT-2|PNAT,MIM:612182|HGNC:HGNC:7646|Ensembl:ENSG00000156...,8,8p22,N-acetyltransferase 2,protein-coding,NAT2,N-acetyltransferase 2,O,arylamine N-acetyltransferase 2|N-acetyltransf...,20250909,-,ENSG00000156006,ENSG00000156006
4,4,9606,11,NATP,-,AACP|NATP1,HGNC:HGNC:15|AllianceGenome:HGNC:15,8,8p22,N-acetyltransferase pseudogene,pseudo,NATP,N-acetyltransferase pseudogene,O,arylamide acetylase pseudogene,20250819,-,NaN,NaN


In [ ]:
homo_gene = pd.read_csv("/Users/kewalinsamart/Documents/GitHub/integrative-drugrep-tb/data/metadata/Homo_sapiens.gene_info.tsv", sep = '\t') # 2025
list(homo_gene.columns)

def extract_ensembl_id(dbxref):
    """
    Extracts the Ensembl ENSG ID from a dbXrefs string.
    Returns the ENSG ID if found, else returns None.
    """
    if pd.isna(dbxref):
        return None
    match = re.search(r'Ensembl:(ENSG[0-9]+)', str(dbxref))
    if match:
        return match.group(1)
    return None

# Apply to DataFrame column
homo_gene['Ensembl_ID'] = homo_gene['dbXrefs'].apply(extract_ensembl_id)
#--- save the file with Ensembl column
homo_gene['Ensembl'] = homo_gene['dbXrefs'].apply(extract_ensembl_id)
homo_gene.to_csv("/Users/kewalinsamart/Documents/GitHub/integrative-drugrep-tb/data/metadata/Homo_sapiens.gene_info_DE.tsv", sep = "\t")

# Build symbol+alias to Ensembl ID mapping
symbol_to_ensg = {}
for _, row in homo_gene.iterrows():
    ensg = row['Ensembl_ID']
    if pd.notna(ensg):
        # Add main symbol
        symbol_to_ensg[row['Symbol']] = ensg
        # Add aliases from Synonyms column (pipe or comma separated)
        if pd.notna(row['Synonyms']):
            for alias in str(row['Synonyms']).replace('|', ',').split(','):
                alias = alias.strip()
                if alias and alias != row['Symbol']:
                    symbol_to_ensg[alias] = ensg

print(f"Total unique symbol to Ensembl ID keys: {len(symbol_to_ensg)}")

symbol_alias = {}

# Build a symbol to alias mapping
for _, row in homo_gene.iterrows():
    symbol = row['Symbol']
    if pd.notna(symbol):
        symbol_alias[symbol] = symbol
        if pd.notna(row['Synonyms']):
            for alias in str(row['Synonyms']).replace('|', ',').split(','):
                alias = alias.strip()
                if alias and alias != symbol:
                    symbol_alias[alias] = symbol

print(f"Total unique symbol to alias keys: {len(symbol_alias)}")


gene_to_type = {}

for _, row in homo_gene.iterrows():
    gene_type = row['type_of_gene']
    ensg = row['Ensembl_ID']
    symbol = row['Symbol']

    # Add ENSG ID
    if pd.notna(ensg):
        gene_to_type[ensg] = gene_type

    # Add main symbol
    if pd.notna(symbol):
        gene_to_type[symbol] = gene_type

    # Add aliases (same type as main symbol)
    if pd.notna(row['Synonyms']):
        for alias in str(row['Synonyms']).replace('|', ',').split(','):
            alias = alias.strip()
            if alias and alias != symbol:
                gene_to_type[alias] = gene_type

print(f"Total unique gene to type keys: {len(gene_to_type):,}")

Total unique symbol to Ensembl ID keys: 101191
Total unique symbol to alias keys: 261748
Total unique gene to type keys: 299,957


In [ ]:
# Get information about the Ensembl_ID column
homo_gene["Ensembl_ID"].nunique()

38209

In [48]:
homo_gene["type_of_gene"].isnull().sum()

np.int64(0)

In [49]:
list(symbol_alias.items())[:10]

[('A1BG', 'A1BG'),
 ('A1B', 'SNTB1'),
 ('ABG', 'A1BG'),
 ('GAB', 'A1BG'),
 ('HYST2477', 'A1BG'),
 ('A2M', 'IGHA2'),
 ('A2MD', 'A2M'),
 ('CPAMD5', 'A2M'),
 ('FWP007', 'A2M'),
 ('S863-7', 'A2M')]

In [50]:
list(symbol_to_ensg.items())[:10]

[('A1BG', 'ENSG00000121410'),
 ('A1B', 'ENSG00000172164'),
 ('ABG', 'ENSG00000121410'),
 ('GAB', 'ENSG00000121410'),
 ('HYST2477', 'ENSG00000121410'),
 ('A2M', 'ENSG00000211890'),
 ('A2MD', 'ENSG00000175899'),
 ('CPAMD5', 'ENSG00000175899'),
 ('FWP007', 'ENSG00000175899'),
 ('S863-7', 'ENSG00000175899')]

In [51]:
list(gene_to_type.items())[:10]

[('ENSG00000121410', 'protein-coding'),
 ('A1BG', 'protein-coding'),
 ('A1B', 'protein-coding'),
 ('ABG', 'protein-coding'),
 ('GAB', 'protein-coding'),
 ('HYST2477', 'protein-coding'),
 ('ENSG00000175899', 'protein-coding'),
 ('A2M', 'other'),
 ('A2MD', 'protein-coding'),
 ('CPAMD5', 'protein-coding')]

In [52]:
# get all the different values in gene_to_type
print(set(gene_to_type.values()))

{'unknown', 'snoRNA', 'scRNA', 'biological-region', 'other', 'protein-coding', 'ncRNA', 'pseudo', 'rRNA', 'snRNA', 'tRNA'}


In [53]:
homo_gene.shape

(193686, 21)

In [54]:
print(homo_gene["type_of_gene"].value_counts())

type_of_gene
biological-region    128261
ncRNA                 22416
protein-coding        20621
pseudo                17494
snoRNA                 1201
unknown                1191
other                   844
rRNA                    785
tRNA                    701
snRNA                   168
scRNA                     4
Name: count, dtype: int64


In [55]:
homo_gene[["Symbol","Ensembl_ID","type_of_gene"]].head()

,Symbol,Ensembl_ID,type_of_gene
0,A1BG,ENSG00000121410,protein-coding
1,A2M,ENSG00000175899,protein-coding
2,NAT1,ENSG00000171428,protein-coding
3,NAT2,ENSG00000156006,protein-coding
4,NATP,NaN,pseudo


In [56]:
proteincoding_HomoGenes = homo_gene.loc[homo_gene["type_of_gene"] == "protein-coding", ["Symbol", "Ensembl_ID", "type_of_gene"]]
print(proteincoding_HomoGenes.shape)
print(proteincoding_HomoGenes.head())

(20621, 3)
     Symbol       Ensembl_ID    type_of_gene
0      A1BG  ENSG00000121410  protein-coding
1       A2M  ENSG00000175899  protein-coding
2      NAT1  ENSG00000171428  protein-coding
3      NAT2  ENSG00000156006  protein-coding
5  SERPINA3  ENSG00000196136  protein-coding


In [57]:
print(proteincoding_HomoGenes.isnull().sum()) # null values are expected not all genes have Ensembl IDs

Symbol             0
Ensembl_ID      1175
type_of_gene       0
dtype: int64


In [ ]:
genetypedf = homo_gene[["Symbol", "Ensembl_ID", "type_of_gene"]]
print(genetypedf.shape)
genetypedf.head()

(193686, 3)


,Symbol,EnsemblID,type_of_gene
0,A1BG,ENSG00000121410,protein-coding
1,A2M,ENSG00000175899,protein-coding
2,NAT1,ENSG00000171428,protein-coding
3,NAT2,ENSG00000156006,protein-coding
4,NATP,None,pseudo


In [59]:
print(genetypedf.isnull().sum())

Symbol               0
EnsemblID       155196
type_of_gene         0
dtype: int64


In [60]:
print(genetypedf["type_of_gene"].value_counts())

type_of_gene
biological-region    128261
ncRNA                 22416
protein-coding        20621
pseudo                17494
snoRNA                 1201
unknown                1191
other                   844
rRNA                    785
tRNA                    701
snRNA                   168
scRNA                     4
Name: count, dtype: int64


## Subset for relevant samples only

In [61]:
def filter_gsm_columns(path: str, AcceptableGSM: set, outputpath: str = None):
    """
    For each .tsv file in 'path', drops columns (GSMs) not in AcceptableGSM.
    If outputpath is provided, saves filtered DataFrames there.
    Otherwise, returns a dict of filtered DataFrames.
    Never edits files in the given path.
    """
    filtered_dfs = {}
    if outputpath and not os.path.exists(outputpath):
        os.makedirs(outputpath)
    for fname in os.listdir(path):
        if fname.endswith('.tsv'):
            fpath = os.path.join(path, fname)
            df = pd.read_csv(fpath, sep='\t')
            gene_col = df.columns[0]
            gsm_cols = [col for col in df.columns[1:] if col in AcceptableGSM]
            keep_cols = [gene_col] + gsm_cols
            filtered_df = df[keep_cols]
            if outputpath:
                out_fpath = os.path.join(outputpath, fname)
                filtered_df.to_csv(out_fpath, sep='\t', index=False)
            else:
                filtered_dfs[fname] = filtered_df
    if not outputpath:
        return filtered_dfs

#### Subest for relevant samples only

In [62]:
filter_gsm_columns(tbdrugrep_expr, Relevant_TB_samples, tbdrugrep_expr)

# Function to remove low quality reads

## Remove low quality reads

* Remove Rows where sum across numeric columns is 0 - Sample is likely an outlier OR technical issue
* Remove Rows where 90% or more values are 0 - Likely a Technical Error 
* Remove Rows where the gene name is missing or NaN -- Gene Name is empty

In [63]:
def remove_low_quality_reads(path, print_shape=False):
    """
    Cleans gene expression files by removing:
    - Rows where sum across numeric columns is 0
    - Rows where 90% or more values are 0
    - Remove Rows where the gene name is missing or NaN
    Args:
        path (str): directory containing .tsv files
        print_shape (bool): if True, prints the shape of each cleaned DataFrame

    Returns:
        dict: cleaned DataFrames keyed by filename
    """
    cleaned_dfs = {}
    files = os.listdir(path)
    for file in files:
        if file.endswith('.tsv'):
            df = pd.read_csv(os.path.join(path, file), sep='\t')
            # Remove rows where the gene name (first column) is missing or NaN
            df = df[df.iloc[:, 0].notna()] # remove rows with NaN in the first column
            numeric_df = df.iloc[:, 1:] # all columns except the first
            threshold = int(0.9 * numeric_df.shape[1]) # 90% of columns
            mask = (numeric_df != 0).sum(axis=1) > threshold 
            mask &= numeric_df.sum(axis=1) != 0
            df_cleaned = df.loc[mask]
            if print_shape:
                print(f"{file}: {df_cleaned.shape}")
            cleaned_dfs[file] = df_cleaned
    return cleaned_dfs

# Function to remap the gene symbols to Ensembl IDs

**to be chained with remove_low_count_reads()**

```python
remapping(remove_low_count_reads())
```

In [64]:
def remapping(cleaned_dfs, output_path=None):
    """
    - Drops "linc" genes
    - Remaps gene using `symbol_to_ensg`
    - Keeps symbol for unmapped genes (no ENSG)
    - Drops duplicates after remapping
    - Optionally saves output
    - Prints a summary for each file

    Returns:
        dict: remapped and cleaned DataFrames
    """
    remapped_dfs = {}

    print(f"\n{'File':<20}{'Post-Filter':>12}{'LINC Dropped':>14}"
          f"{'Unmapped':>10}{'Post-Remap':>12}{'Duplicates':>12}"
          f"{'Final Rows':>12}")
    
    for filename, df in cleaned_dfs.items():
        df = df.copy()
        first_col = df.columns[0]
        pre_remap_rows = df.shape[0]

        # Drop 'linc' genes
        linc_mask = df[first_col].str.lower().str.startswith('linc')
        linc_count = linc_mask.sum()
        df = df.loc[~linc_mask].copy()

        # Remap: keep value if not in dict
        mapped = df[first_col].map(lambda x: symbol_to_ensg.get(x, x))
        unmapped_count = (~df[first_col].isin(symbol_to_ensg.keys())).sum()
        df.loc[:, first_col] = mapped

        # Count duplicates
        duplicate_count = df[first_col].duplicated().sum()
        post_remap_rows = df.shape[0]

        # Drop duplicates
        df = df.drop_duplicates(subset=[first_col])
        final_rows = df.shape[0]

        # Print report line
        print(f"{filename:<20}{pre_remap_rows:>12}{linc_count:>14}"
              f"{unmapped_count:>10}{post_remap_rows:>12}"
              f"{duplicate_count:>12}{final_rows:>12}")

        # Save if path is given
        if output_path:
            output_file = os.path.join(output_path, filename)
            df.to_csv(output_file, sep='\t', index=False)

        remapped_dfs[filename] = df

    return remapped_dfs

# Process all the expression data files

In [65]:
TBcleaned = remapping(remove_low_quality_reads(tbdrugrep_expr), tbdrugrep_expr)


File                 Post-Filter  LINC Dropped  Unmapped  Post-Remap  Duplicates  Final Rows
GSE42834.tsv               23374             0     23374       23374           0       23374
GSE34151.tsv               23374             0     23374       23374           0       23374
GSE42830.tsv               23374             0     23374       23374           0       23374
GSE16250.tsv               20056             0     20056       20056           0       20056
GSE139871.tsv              23552             0     23552       23552           0       23552
GSE18794.tsv               21661             0     21661       21661           0       21661
GSE39941.tsv                   0             0         0           0           0           0
GSE119143.tsv              35066             0     35066       35066           0       35066
GSE19491.tsv               21799             0     21799       21799           0       21799
GSE83456.tsv               23374             0     23374       23374 

In [66]:
get_expression_shape(tbdrugrep_expr)

Number of studies in expression_matrices: 12
GSE42834.tsv: (23374, 8)
GSE34151.tsv: (23374, 260)
GSE42830.tsv: (23374, 8)
GSE16250.tsv: (20056, 7)
GSE139871.tsv: (23552, 25)
GSE18794.tsv: (21661, 10)
GSE39941.tsv: (0, 1)
GSE119143.tsv: (35066, 13)
GSE19491.tsv: (21799, 17)
GSE83456.tsv: (23374, 154)
GSE57028.tsv: (0, 1)
GSE63548.tsv: (23371, 27)


## Subset for protein coding genes only

In [30]:
# protein coding gene only
def filter_protein_coding_genes(expressionpath : str, gene_to_type: Dict[str, str], save : bool = False) -> pd.DataFrame:
	"""
	Given a path to an folder containing expression .tsv files, where dims are gene x samples,
	perform filtering to keep only protein coding genes based on gene_to_type mapping.

	if save is True, saves each DataFrames back to the same files, else returns a dictionary of filtered DataFrames.
	
	"""
	files = os.listdir(expressionpath)
	filtered_dfs = {}
	for file in files:
		if file.endswith('.tsv'):
			fpath = os.path.join(expressionpath, file)
			df = pd.read_csv(fpath, sep = '\t')
			gene_col = df.columns[0]
			# Identify protein-coding genes
			protein_coding_genes = [
				gene for gene in df[gene_col]
				if gene in gene_to_type and gene_to_type[gene] == 'protein-coding'
			]
			# Filter DataFrame
			filtered_df = df[df[gene_col].isin(protein_coding_genes)]
			if save:
				filtered_df.to_csv(fpath, sep = '\t', index = False)
			else:
				filtered_dfs[file] = filtered_df
	if not save:
		return filtered_dfs

#### Filter and save protein coding genes only

In [31]:
TB_filteredpc = filter_protein_coding_genes(tbdrugrep_expr, gene_to_type, save = True)

In [32]:
get_expression_shape(tbdrugrep_expr)

Number of studies in expression_matrices: 12
GSE42834.tsv: (18590, 8)
GSE34151.tsv: (18590, 260)
GSE42830.tsv: (18590, 8)
GSE16250.tsv: (16082, 7)
GSE139871.tsv: (18623, 25)
GSE18794.tsv: (17215, 10)
GSE39941.tsv: (0, 1)
GSE119143.tsv: (18158, 13)
GSE19491.tsv: (18217, 17)
GSE83456.tsv: (18590, 154)
GSE57028.tsv: (0, 1)
GSE63548.tsv: (18589, 27)


In [46]:
get_expression_shape("../data/FinalExpression/TB") # comparing to xlung

Number of studies in TB: 21
GSE236156.tsv: (13400, 36)
GSE164287.tsv: (12883, 9)
GSE148731.tsv: (12671, 25)
GSE183912.tsv: (11852, 9)
GSE132283.tsv: (15459, 19)
GSE165708.tsv: (13311, 33)
GSE143731.tsv: (13432, 9)
GSE174566.tsv: (14898, 37)
GSE256184.tsv: (15717, 17)
GSE141656.tsv: (13517, 13)
GSE223863.tsv: (13642, 29)
GSE193777.tsv: (13550, 47)
GSE148171.tsv: (14319, 10)
GSE64179.tsv: (14879, 13)
GSE198557.tsv: (14815, 13)
GSE143627.tsv: (13409, 7)
GSE112483.tsv: (9981, 84)
GSE121049.tsv: (12625, 7)
GSE84076.tsv: (13417, 11)
GSE64182.tsv: (14528, 7)
GSE112482.tsv: (2865, 88)


## Verify Integrity

* get a count of genes vs Ensemble IDs
* make sure there are no duplicate Ensemble IDs

In [43]:
def verify_process(path: str):
    """
    For each .tsv file in the given path:
    - Prints the total number of genes (rows)
    - Prints the number of Ensembl IDs (rows where first col starts with 'ENSG')
    - Prints the number of gene symbols (rows where first col does NOT start with 'ENSG')
    - Checks for duplicate Ensembl IDs
    - Checks for duplicate gene names (using symbol_alias mapping)
    - Prints if any file has duplicates before printing details
    """
    files = [f for f in os.listdir(path) if f.endswith('.tsv')]
    has_ensg_duplicates = False
    has_symbol_duplicates = False
    for file in files:
        df = pd.read_csv(os.path.join(path, file), sep='\t')
        gene_col = df.columns[0]
        ensg_mask = df[gene_col].astype(str).str.startswith('ENSG')
        # Check for duplicate Ensembl IDs
        if df.loc[ensg_mask, gene_col].duplicated().any():
            has_ensg_duplicates = True
        # Check for duplicate gene names (map all to canonical symbol, then check dups)
        mapped_symbols = df.loc[~ensg_mask, gene_col].map(lambda x: symbol_alias.get(x, x))
        if mapped_symbols.duplicated().any():
            has_symbol_duplicates = True
    
    print("Duplicates present in any file:" if has_ensg_duplicates or has_symbol_duplicates else "No duplicates found in any file.")
    if has_ensg_duplicates:
        print("- Duplicate Ensembl IDs found.")
    if has_symbol_duplicates:
        print("- Duplicate gene names (canonical symbol) found.")
    
    print(f"{'File':<20}{'Total Genes':>12}{'Ensembl IDs':>14}{'Gene Symbols':>14}{'ENSG Duplicates':>16}{'Symbol Duplicates':>18}")
    for file in files:
        df = pd.read_csv(os.path.join(path, file), sep='\t')
        gene_col = df.columns[0]
        total_genes = df.shape[0]
        ensg_mask = df[gene_col].astype(str).str.startswith('ENSG')
        ensg_count = ensg_mask.sum()
        symbol_count = total_genes - ensg_count
        ensg_duplicates = df.loc[ensg_mask, gene_col].duplicated().sum()
        mapped_symbols = df.loc[~ensg_mask, gene_col].map(lambda x: symbol_alias.get(x, x))
        symbol_duplicates = mapped_symbols.duplicated().sum()
        print(f"{file:<20}{total_genes:>12}{ensg_count:>14}{symbol_count:>14}{ensg_duplicates:>16}{symbol_duplicates:>18}")

In [44]:
verify_process(tbdrugrep_expr)

No duplicates found in any file.
File                 Total Genes   Ensembl IDs  Gene Symbols ENSG Duplicates Symbol Duplicates
GSE42834.tsv               18590         18590             0               0                 0
GSE34151.tsv               18590         18590             0               0                 0
GSE42830.tsv               18590         18590             0               0                 0
GSE16250.tsv               16082         16082             0               0                 0
GSE139871.tsv              18623         18623             0               0                 0
GSE18794.tsv               17215         17215             0               0                 0
GSE39941.tsv               18590         18590             0               0                 0
GSE119143.tsv              18158         18158             0               0                 0
GSE19491.tsv               18217         18217             0               0                 0
GSE83456.tsv     